# Analytics Hub - Load Status

The last step of the Analytics Hub pipeline. It reads how every step of **this run** went and
writes one row per source to the Delta table **`dbo.load_log`**, in plain words:

| Column | What it holds |
|---|---|
| `run_id` | The pipeline run |
| `activity`, `source` | The pipeline step and the source it loads, e.g. `Run_Licensed_Users_Ingester`, *Licensed users* |
| `status` | Succeeded, Failed, Cancelled, Skipped (waited on a step that failed) |
| `reason_kind`, `reason` | Why it failed: `capacity` (Fabric was too busy), `signIn`, `timeout`, `noData` or `other` |
| `attempts` | How many times the pipeline tried it |
| `error_code`, `error_message` | Fabric's own error, for support |

Rows for the same run are replaced, so running it again for a run is safe.

When any load failed it fails too, after writing the log, so the pipeline run shows as failed.
A failed Agent 365 API load covered by the Agent 365 export doesn't count.

To rerun just the failed loads, run `AnalyticsHubInstaller.exe rerun-failed`, or choose **Rerun failed** in the installer.

In [ ]:
# === CONFIG ===
PIPELINE_RUN_ID       = ''               # set by the pipeline: @pipeline().RunId
PIPELINE_TRIGGER_TIME = ''               # set by the pipeline: @pipeline().TriggerTime
LOG_TABLE             = 'dbo.load_log'   # one row per load per run
STATUS_JSON           = '{"labels":{"Run_Audit_Log_Ingester":"Copilot audit log","Run_Audit_Log_Processor":"Copilot usage (audit log processing)","Refresh_Semantic_Model":"ValueLens model refresh","Run_Licensed_Users_Ingester":"Licensed users","Run_Upload_Router":"Uploaded CSV exports","Run_Agent365_Registry_Ingester":"Agent 365 (API)","Run_Agent365_CSV_Fallback":"Agent 365 (export)","Run_Agent365_Lander":"Agent 365 (export)","Run_Org_Data_Ingester":"Org data (Entra ID)","Run_Org_Data_Workday":"Org data (Workday export)","Run_M365_Activity_Ingester":"Microsoft 365 activity","Run_ProductFeedback_Ingester":"Product feedback","Run_Consumption_Azure_AI":"Azure AI spend","Run_Consumption_Studio":"Copilot Studio credits","Refresh_Cowork_Credits":"Cowork credits (Viva Insights)","Run_Consumption_Viva":"Cowork credits (load)","Refresh_Consumption_Model":"Consumption model refresh","Run_Agent_Evaluator_Transcripts":"Agent transcripts","Refresh_Agent_Evaluator_Model":"Agent Evaluator model refresh","Run_Resource_Graph_Ingester":"Agent configuration and Foundry (Resource Graph)","Record_Load_Status":"Load status record"},"reasons":[{"kind":"capacity","pattern":"TooManyRequestsForCapacity|(?:code|status)[\\\\s:\'\\"=-]{0,6}430\\\\b","text":"Fabric\'s capacity was too busy to start it."},{"kind":"notSynced","pattern":"is not in (?:the )?database|SQL (?:analytics )?endpoint (?:to|has not|hasn.t) (?:sync|caught up)","text":"The model refreshed before the Lakehouse\'s SQL endpoint had caught up with the new tables."},{"kind":"signIn","pattern":"AADSTS\\\\d+|\\\\b401\\\\b|\\\\b403\\\\b|Unauthori[sz]ed|Forbidden|invalid_client|consent|credential|sign.?in","text":"It couldn\'t sign in, or wasn\'t allowed to read the data."},{"kind":"timeout","pattern":"timed? ?out|timeout|exceeded the (?:maximum|allowed)","text":"It ran out of time."},{"kind":"noData","pattern":"PATH_NOT_FOUND|Path does not exist|FileNotFound|No such file","text":"The file it reads wasn\'t there."}]}'

In [ ]:
# === HELPERS (pure, so they can be tested off Fabric) ===
import json
import re
from datetime import datetime, timedelta, timezone

SELF = 'Record_Load_Status'
REGISTRY = 'Run_Agent365_Registry_Ingester'
FALLBACK = 'Run_Agent365_CSV_Fallback'
FAILED = ('Failed', 'Cancelled', 'TimedOut')
GUID = re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')


def label(name, labels):
    if name in labels:
        return labels[name]
    text = re.sub(r'^Run_', '', name)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    return text.replace('_', ' ')


def classify(error, reasons):
    # Same order and patterns as the installer's loads.js, so both say the same thing.
    error = error or {}
    subject = f"errorCode={error.get('errorCode') or ''} {error.get('message') or ''}"
    for r in reasons:
        if re.search(r['pattern'], subject, re.IGNORECASE):
            return r['kind'], r['text']
    first = next((l.strip() for l in str(error.get('message') or '').splitlines() if l.strip()), '')
    return 'other', (f'It stopped with an error: {first[:240]}' if first else 'It stopped without saying why.')


def parse_time(text):
    # Pipeline times carry up to 7 fractional digits, which older Pythons can't parse; seconds are enough.
    m = re.match(r'^\s*(\d{4}-\d{2}-\d{2})[T ](\d{2}:\d{2}:\d{2})', str(text or ''))
    if not m:
        return None
    return datetime.strptime(f'{m.group(1)}T{m.group(2)}', '%Y-%m-%dT%H:%M:%S').replace(tzinfo=timezone.utc)


def window(trigger_time, now):
    start = parse_time(trigger_time) or now - timedelta(days=2)
    return start - timedelta(hours=1), now + timedelta(hours=1)


def last_attempts(runs):
    # The last attempt of each step, with how many attempts there were.
    last = {}
    for r in sorted(runs, key=lambda r: str(r.get('activityRunStart') or '')):
        name = r.get('activityName')
        prev = last.get(name)
        last[name] = dict(r, attempts=(prev['attempts'] if prev else 0) + 1)
    return last


def is_load(r):
    return r.get('activityName') != SELF and r.get('activityType') != 'IfCondition'


def failed_loads(last):
    status = {n: r.get('status') for n, r in last.items()}
    out = []
    for name, r in last.items():
        if not is_load(r) or r.get('status') not in FAILED:
            continue
        if name == REGISTRY and status.get(FALLBACK) == 'Succeeded':
            continue
        if name == FALLBACK and status.get(REGISTRY) == 'Succeeded':
            continue
        out.append(name)
    return out


def rows(last, labels, reasons, run_id, logged_at):
    out = []
    for name, r in last.items():
        if not is_load(r):
            continue
        status = r.get('status')
        error = r.get('error') or {}
        kind, reason = classify(error, reasons) if status in FAILED else (None, None)
        ms = r.get('durationInMs')
        out.append({
            'run_id': run_id,
            'activity': name,
            'source': label(name, labels),
            'activity_type': r.get('activityType'),
            'status': status,
            'reason_kind': kind,
            'reason': reason,
            'attempts': int(r.get('attempts') or 1),
            'started_at': parse_time(r.get('activityRunStart')),
            'ended_at': parse_time(r.get('activityRunEnd')),
            'duration_seconds': None if ms is None else round(float(ms) / 1000, 1),
            'error_code': (str(error.get('errorCode')) or None) if error.get('errorCode') else None,
            'error_message': (str(error.get('message'))[:4000] or None) if error.get('message') else None,
            'logged_at': logged_at,
        })
    return out

In [ ]:
# === RECORD ===
import notebookutils
import sempy.fabric as fabric
from pyspark.sql import types as T

STATUS = json.loads(STATUS_JSON)
NOW = datetime.now(timezone.utc)

SCHEMA = T.StructType([
    T.StructField('run_id', T.StringType()), T.StructField('activity', T.StringType()),
    T.StructField('source', T.StringType()), T.StructField('activity_type', T.StringType()),
    T.StructField('status', T.StringType()), T.StructField('reason_kind', T.StringType()),
    T.StructField('reason', T.StringType()), T.StructField('attempts', T.IntegerType()),
    T.StructField('started_at', T.TimestampType()), T.StructField('ended_at', T.TimestampType()),
    T.StructField('duration_seconds', T.DoubleType()), T.StructField('error_code', T.StringType()),
    T.StructField('error_message', T.StringType()), T.StructField('logged_at', T.TimestampType()),
])


def query_runs(workspace_id, run_id):
    after, before = window(PIPELINE_TRIGGER_TIME, NOW)
    client = fabric.FabricRestClient()
    path = f'v1/workspaces/{workspace_id}/datapipelines/pipelineruns/{run_id}/queryactivityruns'
    body = {'filters': [], 'orderBy': [{'orderBy': 'ActivityRunStart', 'order': 'ASC'}],
            'lastUpdatedAfter': after.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'lastUpdatedBefore': before.strftime('%Y-%m-%dT%H:%M:%SZ')}
    out = []
    for _ in range(50):
        resp = client.post(path, json=body)
        if resp.status_code >= 400:
            raise RuntimeError(f'Could not read the pipeline run ({resp.status_code}): {resp.text[:500]}')
        data = resp.json()
        out.extend(data if isinstance(data, list) else data.get('value', []))
        token = None if isinstance(data, list) else data.get('continuationToken')
        if not token:
            break
        body['continuationToken'] = token
    return out


def record():
    if not GUID.match(PIPELINE_RUN_ID or ''):
        print('No pipeline run to record: this notebook runs as the last step of the Analytics Hub pipeline.')
        return []
    workspace_id = notebookutils.runtime.context['currentWorkspaceId']
    last = last_attempts(query_runs(workspace_id, PIPELINE_RUN_ID))
    data = rows(last, STATUS['labels'], STATUS['reasons'], PIPELINE_RUN_ID, NOW.replace(tzinfo=None))
    for r in data:
        for k in ('started_at', 'ended_at'):
            if r[k] is not None:
                r[k] = r[k].replace(tzinfo=None)
    if spark.catalog.tableExists(LOG_TABLE):
        spark.sql(f"DELETE FROM {LOG_TABLE} WHERE run_id = '{PIPELINE_RUN_ID}'")
    (spark.createDataFrame([tuple(r[f.name] for f in SCHEMA.fields) for r in data], SCHEMA)
        .write.format('delta').mode('append').saveAsTable(LOG_TABLE))
    for r in data:
        mark = {'Succeeded': 'ok  ', 'Skipped': 'skip'}.get(r['status'], 'FAIL')
        print(f"{mark} {r['source']}" + (f": {r['reason']}" if r['reason'] else ''))
    return [label(n, STATUS['labels']) for n in failed_loads(last)]


FAILED_LOADS = record()
if FAILED_LOADS:
    raise RuntimeError(f"{len(FAILED_LOADS)} load(s) failed: {', '.join(FAILED_LOADS)}. "
                       f"See {LOG_TABLE} for why, then run 'AnalyticsHubInstaller.exe rerun-failed'.")